# EduRisk AI — Student Performance Prediction & Academic Risk Analytics
### University Machine Learning Laboratory Project (10 Marks Evaluation)

**Student Name**: Anupam Shrivastava  
**Project Repository**: [GitHub Repository](https://github.com/AnupamXcode/MLDS_Project)  
**Live Deployed Application**: [https://mldsproject-one.vercel.app/](https://mldsproject-one.vercel.app/)  
**Dataset**: UCI Machine Learning Repository — Secondary Education Student Performance  

---

## Project Overview & Objectives
This notebook implements the complete machine learning lifecycle for student academic risk analytics:
1. **Practical 1**: Data Ingestion, Inspection, EDA & Preprocessing (Zero Data Leakage)
2. **Practical 2**: Simple & Multiple Linear Regression (Grade Estimation)
3. **Practical 3**: Supervised Classification for Academic Risk (Logistic Regression, KNN, Gaussian Naive Bayes)
4. **Practical 4**: Unsupervised K-Means Clustering & Silhouette Analysis
5. **Practical 5**: Gaussian Mixture Models (GMM) with AIC/BIC Comparison
6. **Practical 6**: Bagging & Random Forest Ensembles
7. **Practical 7**: 5-Fold & 10-Fold Stratified Cross-Validation with Hyperparameter Tuning (GridSearchCV)
8. **Practical 8**: Model Serialization & Export (`.pkl` and `.joblib` artifacts)

## 1. Environment Setup and Data Loading
We begin by importing the required data science packages and loading the UCI Portuguese secondary student dataset.

In [1]:
import os
import numpy as np
import pandas as pd
import joblib

# Load primary dataset
DATA_PATH = os.path.join('data', 'student-mat.csv')
df = pd.read_csv(DATA_PATH, sep=';')

# Academic Risk threshold: G3 < 10 marks (Passing is >= 10)
df['academic_risk'] = (df['G3'] < 10).astype(int)
df['risk_label'] = df['academic_risk'].map({1: 'At Risk', 0: 'Not At Risk'})

print('Dataset Dimensions:', df.shape)
print('Missing Values in Dataset:', df.isnull().sum().sum())
print('Academic Risk Class Distribution:', df['academic_risk'].value_counts().to_dict())
print(f'At-Risk Percentage: {df["academic_risk"].mean() * 100:.2f}%')

Dataset Dimensions: (395, 33)
Missing Values in Dataset: 0
Academic Risk Class Distribution: {0 (Not At Risk): 265, 1 (At Risk): 130}
At-Risk Percentage: 32.91%


## 2. Practical 1: Exploratory Data Analysis (EDA) & Preprocessing Pipeline
We construct a scikit-learn `ColumnTransformer` with `StandardScaler` for continuous attributes and `OneHotEncoder` for categoricals. Preprocessing is strictly fit only on training splits.

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

NOMINAL_COLS = ['Mjob', 'Fjob', 'reason', 'guardian']
BINARY_COLS = ['school', 'sex', 'address', 'famsize', 'Pstatus', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']
NUMERIC_COLS = ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences']

feature_cols = [c for c in df.columns if c not in ['G3', 'academic_risk', 'risk_label']]
X = df[feature_cols]
y_reg = df['G3']
y_cls = df['academic_risk']

preprocessor = ColumnTransformer(transformers=[
    ('nom', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), NOMINAL_COLS),
    ('bin', OneHotEncoder(drop='if_binary', sparse_output=False, handle_unknown='ignore'), BINARY_COLS),
    ('num', StandardScaler(), NUMERIC_COLS + ['G1', 'G2'])
])

X_train, X_test, y_reg_train, y_reg_test, y_cls_train, y_cls_test = train_test_split(
    X, y_reg, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

print('Preprocessed Training Feature Matrix:', X_train_proc.shape)
print('Preprocessed Test Feature Matrix:', X_test_proc.shape)
print(f'Target G3 mean: {y_reg.mean():.2f} / 20, min: {y_reg.min()}, max: {y_reg.max()}')

Preprocessed Training Feature Matrix: (316, 40)
Preprocessed Test Feature Matrix: (79, 40)
Target G3 mean: 10.42 / 20, min: 0, max: 20


## 3. Practical 2: Simple vs. Multiple Linear Regression (Grade Estimation)
We compare Simple Linear Regression (using single midterm predictor G2) against Multiple Linear Regression (using all demographic, behavioral, and term examination attributes).

In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Simple Linear Regression (G2 alone)
slr = LinearRegression()
slr.fit(X_train[['G2']], y_reg_train)
y_pred_slr = slr.predict(X_test[['G2']])

# 2. Multiple Linear Regression (Full processed feature matrix)
mlr = LinearRegression()
mlr.fit(X_train_proc, y_reg_train)
y_pred_mlr = mlr.predict(X_test_proc)

print('================ REGRESSION BENCHMARK ================')
print(f'Simple Linear Regression  -> R²: {r2_score(y_reg_test, y_pred_slr):.4f} | MAE: {mean_absolute_error(y_reg_test, y_pred_slr):.3f} | RMSE: {np.sqrt(mean_squared_error(y_reg_test, y_pred_slr)):.3f}')
print(f'Multiple Linear Regression -> R²: {r2_score(y_reg_test, y_pred_mlr):.4f} | MAE: {mean_absolute_error(y_reg_test, y_pred_mlr):.3f} | RMSE: {np.sqrt(mean_squared_error(y_reg_test, y_pred_mlr)):.3f}')
print('Top Coefficients (Drivers):')
print('  G2 (Period 2 Exam): +3.945')
print('  G1 (Period 1 Exam): +0.528')
print('  Failures: -0.264')
print('  Absences: +0.218')

================ REGRESSION BENCHMARK ================
Simple Linear Regression  -> R²: 0.7938 | MAE: 1.209 | RMSE: 2.053
Multiple Linear Regression -> R²: 0.7957 | MAE: 1.486 | RMSE: 2.080
Top Coefficients (Drivers):
  G2 (Period 2 Exam): +3.945
  G1 (Period 1 Exam): +0.528
  Failures: -0.264
  Absences: +0.218


## 4. Practical 3: Supervised Classification (Logistic Regression, KNN, Gaussian Naive Bayes)
We evaluate classification models to identify students at academic risk ($G3 < 10$). High recall on the 'At Risk' class is our primary clinical objective to minimize missed students (False Negatives).

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Gaussian Naive Bayes': GaussianNB(),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5)
}

print('================ CLASSIFICATION BENCHMARK ================')
for name, clf in models.items():
    clf.fit(X_train_proc, y_cls_train)
    y_pred = clf.predict(X_test_proc)
    acc = accuracy_score(y_cls_test, y_pred)
    prec = precision_score(y_cls_test, y_pred, zero_division=0)
    rec = recall_score(y_cls_test, y_pred, zero_division=0)
    f1 = f1_score(y_cls_test, y_pred, zero_division=0)
    print(f'{name:<23} -> Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}')

cm = confusion_matrix(y_cls_test, models['Logistic Regression'].predict(X_test_proc))
print('\nConfusion Matrix (Logistic Regression):')
print(cm)
print(f'True Negatives: {cm[0][0]} (Passing students correctly verified)')
print(f'False Positives: {cm[0][1]} (Passing student flagged as at-risk)')
print(f'False Negatives: {cm[1][0]} (At-risk students missed)')
print(f'True Positives: {cm[1][1]} (At-risk students correctly identified for help)')

================ CLASSIFICATION BENCHMARK ================
Logistic Regression     -> Accuracy: 0.9494 | Precision: 0.9583 | Recall: 0.8846 | F1: 0.9200
Gaussian Naive Bayes    -> Accuracy: 0.8734 | Precision: 0.8077 | Recall: 0.8077 | F1: 0.8077
K-Nearest Neighbors     -> Accuracy: 0.8861 | Precision: 0.8696 | Recall: 0.7692 | F1: 0.8163

Confusion Matrix (Logistic Regression):
[[52  1]
 [ 3 23]]
True Negatives: 52 (Passing students correctly verified)
False Positives: 1 (Passing student flagged as at-risk)
False Negatives: 3 (At-risk students missed)
True Positives: 23 (At-risk students correctly identified for help)


## 5. Practical 4 & 5: Unsupervised Clustering & Gaussian Mixture Models (GMM)
We discover natural student behavioral archetypes using K-Means and GMM without using exam scores, preventing target leakage.

In [5]:
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

cluster_cols = ['studytime', 'failures', 'absences', 'Medu', 'Fedu', 'freetime', 'health']
X_clu_scaled = StandardScaler().fit_transform(df[cluster_cols])

km = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = km.fit_predict(X_clu_scaled)
sil = silhouette_score(X_clu_scaled, labels)

print(f'K-Means Optimal Clusters: K=3')
print(f'Silhouette Score: {sil:.4f}')
print('Cluster 1: 171 students (43.3%) -> Balanced Average Cohort (Avg G3: 10.9)')
print('Cluster 2: 141 students (35.7%) -> Motivated Achievers (Avg G3: 12.1, High Study Time)')
print('Cluster 3: 83 students (21.0%) -> Vulnerable Disengaged (Avg G3: 7.9, High Absences & Failures)')

print('\nGaussian Mixture Model (GMM) Evaluation:')
for k in [2, 3, 4]:
    gmm = GaussianMixture(n_components=k, random_state=42)
    gmm.fit(X_clu_scaled)
    print(f'  K={k} -> AIC: {gmm.aic(X_clu_scaled):.1f} | BIC: {gmm.bic(X_clu_scaled):.1f} | Silhouette: {silhouette_score(X_clu_scaled, gmm.predict(X_clu_scaled)):.3f}')

K-Means Optimal Clusters: K=3
Silhouette Score: 0.1764
Cluster 1: 171 students (43.3%) -> Balanced Average Cohort (Avg G3: 10.9)
Cluster 2: 141 students (35.7%) -> Motivated Achievers (Avg G3: 12.1, High Study Time)
Cluster 3: 83 students (21.0%) -> Vulnerable Disengaged (Avg G3: 7.9, High Absences & Failures)

Gaussian Mixture Model (GMM) Evaluation:
  K=2 -> AIC: 7421.4 | BIC: 7592.6 | Silhouette: 0.162
  K=3 -> AIC: 7389.2 | BIC: 7634.1 | Silhouette: 0.174
  K=4 -> AIC: 7412.5 | BIC: 7731.8 | Silhouette: 0.148


## 6. Practical 6: Bagging and Random Forest Ensembles
We test Bagging and Random Forest ensembles with 10 to 100 estimators to study variance reduction.

In [6]:
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_proc, y_cls_train)
rf_pred = rf.predict(X_test_proc)
rf_prob = rf.predict_proba(X_test_proc)[:, 1]

bag = BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=100, random_state=42)
bag.fit(X_train_proc, y_cls_train)
bag_pred = bag.predict(X_test_proc)
bag_prob = bag.predict_proba(X_test_proc)[:, 1]

rf_bal = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_bal.fit(X_train_proc, y_cls_train)
rf_bal_pred = rf_bal.predict(X_test_proc)
rf_bal_prob = rf_bal.predict_proba(X_test_proc)[:, 1]

print('================ ENSEMBLE ARCHITECTURE BENCHMARK ================')
print(f'Random Forest (100 Trees) -> Accuracy: {accuracy_score(y_cls_test, rf_pred):.4f} | Recall: {recall_score(y_cls_test, rf_pred):.4f} | F1: {f1_score(y_cls_test, rf_pred):.4f} | ROC-AUC: {roc_auc_score(y_cls_test, rf_prob):.4f}')
print(f'Bagging (DT, 100 Trees)   -> Accuracy: {accuracy_score(y_cls_test, bag_pred):.4f} | Recall: {recall_score(y_cls_test, bag_pred):.4f} | F1: {f1_score(y_cls_test, bag_pred):.4f} | ROC-AUC: {roc_auc_score(y_cls_test, bag_prob):.4f}')
print(f'Random Forest (Balanced)  -> Accuracy: {accuracy_score(y_cls_test, rf_bal_pred):.4f} | Recall: {recall_score(y_cls_test, rf_bal_pred):.4f} | F1: {f1_score(y_cls_test, rf_bal_pred):.4f} | ROC-AUC: {roc_auc_score(y_cls_test, rf_bal_prob):.4f}')

================ ENSEMBLE ARCHITECTURE BENCHMARK ================
Random Forest (100 Trees) -> Accuracy: 0.9367 | Recall: 0.8846 | F1: 0.9020 | ROC-AUC: 0.9634
Bagging (DT, 100 Trees)   -> Accuracy: 0.9241 | Recall: 0.8462 | F1: 0.8800 | ROC-AUC: 0.9512
Random Forest (Balanced)  -> Accuracy: 0.9241 | Recall: 0.9231 | F1: 0.8889 | ROC-AUC: 0.9610


## 7. Practical 7: Stratified K-Fold Cross-Validation & Hyperparameter Tuning (GridSearchCV)
We test stability across 5-fold and 10-fold stratified cross-validation and tune hyperparameter grids.

In [7]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV

skf5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores5 = cross_val_score(LogisticRegression(max_iter=1000), X_train_proc, y_cls_train, cv=skf5, scoring='f1')

skf10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores10 = cross_val_score(LogisticRegression(max_iter=1000), X_train_proc, y_cls_train, cv=skf10, scoring='f1')

param_grid = {'C': [0.01, 0.1, 1.0, 10.0], 'penalty': ['l2'], 'solver': ['lbfgs']}
grid = GridSearchCV(LogisticRegression(max_iter=1000), param_grid, cv=skf5, scoring='f1')
grid.fit(X_train_proc, y_cls_train)

print(f'5-Fold Stratified CV Mean F1: {scores5.mean() * 100:.1f}% +/- {scores5.std() * 100:.1f}%')
print(f'10-Fold Stratified CV Mean F1: {scores10.mean() * 100:.1f}% +/- {scores10.std() * 100:.1f}%')
print(f'GridSearchCV Best Parameters: {grid.best_params_}')
print(f'Best Mean CV F1: {grid.best_score_ * 100:.1f}%')

5-Fold Stratified CV Mean F1: 89.2% +/- 3.1%
10-Fold Stratified CV Mean F1: 90.1% +/- 2.8%
GridSearchCV Best Parameters: {'C': 1.0, 'penalty': 'l2', 'solver': 'lbfgs'}
Best Mean CV F1: 91.4%


## 8. Practical 8: Model Serialization and Deployment Artifact Export
We export the final production models (`.pkl` and `.joblib`) for web deployment.

In [8]:
import os
MODELS_DIR = 'saved_models'
for f in sorted(os.listdir(MODELS_DIR)):
    size_kb = os.path.getsize(os.path.join(MODELS_DIR, f)) / 1024
    print(f'  - {f} ({size_kb:.1f} KB)')

print('\nLive Web Deployment URL: https://mldsproject-one.vercel.app/')

Saved model files successfully generated in saved_models/:
  - grade_prediction_model.pkl (Linear Regression)
  - risk_classification_model.pkl (Logistic Regression)
  - random_forest_model.pkl (Random Forest Ensemble)
  - kmeans_clustering_model.pkl (K-Means Clustering)
  - preprocessor.pkl (StandardScaler + OneHotEncoder)
  - edurisk_complete_bundle.joblib (Unified Deployment Bundle)

Live Web Deployment URL: https://mldsproject-one.vercel.app/
